# 01 · MegaDetector boxes (Crop-CLIP)

**Plan:** `crop-clip/experiment-plan.md`, decision D2 and task T3 · **Needs:** notebook 00 run once
(dataset on this machine; image statistics optional)

What this notebook does:
1. Downloads **MegaDetector v5a** from the official release and checks the weights' MD5.
2. Runs it on **all 203,029 images** (every split: this is inference only, no labels are used), in
   resumable chunks, keeping every box with confidence ≥ 0.005. The real cutoff is chosen later on
   OOD val.
3. Merges the chunks into one table and uploads it as the W&B artifact `megadetector-v5a-boxes`.
4. Box statistics and preview grids (plan section 5.1, item 7) on **train / ID val / OOD val only**.

**Splitting the run across teammates:** set `SHARD = "0/3"`, `"1/3"`, `"2/3"` on three machines.
Each machine processes every third chunk and uploads its own shard artifact; the merge cell on any
machine downloads the missing shards and builds the full table. One machine: keep `"0/1"`.

**Resuming:** finished chunks are files on disk, so rerunning the run cell skips them. Molab sessions
stop after 12 h, so keep an eye on the timing estimate.

Output table (`boxes_all.csv.gz`), one row per box: `filename, det_rank, category, conf, x, y, w, h,
failure` with `[x, y, w, h]` normalised to [0, 1] (MegaDetector's format). A photo with no box keeps
one row with `det_rank = -1`, so every photo appears.

In [ ]:
# Setup: shared code (crop-clip/common.py from the team repo) + packages + all imports
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time

REPO_URL = "https://github.com/rnjema/idl-f26-g30-project.git"
REPO_BRANCH = "main"
REPO_DIR = Path(os.environ.get("CROPCLIP_REPO_DIR", Path.cwd() / "idl-f26-g30-project"))
PACKAGES = ["megadetector", "huggingface_hub", "wandb", "pandas", "matplotlib", "pillow", "tqdm"]

if (Path.cwd() / "common.py").is_file():  # notebook opened inside crop-clip/ of a clone
    CODE_DIR = Path.cwd()
else:  # Molab: clone (or update) the team repo next to the notebook
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True
        )
    CODE_DIR = REPO_DIR / "crop-clip"
if os.environ.get("CROPCLIP_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
sys.path.insert(0, str(CODE_DIR))

from importlib import metadata as pkg_metadata

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm

import common as cc

importlib.reload(cc)
print("shared code:", CODE_DIR / "common.py", "| git commit:", cc.git_commit(CODE_DIR))
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| megadetector", pkg_metadata.version("megadetector"))

## Settings and keys

Paste your W&B API key **only in your private Molab copy** (or set `WANDB_API_KEY`). Put the
placeholder back before sharing or committing.

In [ ]:
WANDB_API_KEY = "PASTE_YOUR_WANDB_API_KEY_HERE"  # private copy only; never commit a real key
USE_WANDB = os.environ.get("CROPCLIP_USE_WANDB", "1") == "1"
WANDB_BASE_URL = "https://api.forge.coreweave.com"  # team W&B server (CoreWeave Forge)
SHARD = os.environ.get("CROPCLIP_SHARD", "0/1")  # "i/n": this machine runs every n-th chunk
BATCH_SIZE = 16  # images per GPU batch at 1280 px; lower it if you run out of GPU memory
LOADER_THREADS = 8  # threads decoding JPEGs while the GPU works
CHUNK_SIZE = int(os.environ.get("CROPCLIP_CHUNK_SIZE", "2000"))  # images per resumable chunk file
TIMING_IMAGES = int(os.environ.get("CROPCLIP_TIMING_IMAGES", "256"))  # photos for the speed test
MAX_IMAGES = int(os.environ.get("CROPCLIP_MAX_IMAGES", "0"))  # 0 = all images (>0 only for tests)
REQUIRE_GPU = os.environ.get("CROPCLIP_REQUIRE_GPU", "1") == "1"
SEED = 0

if "PASTE_YOUR" not in WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
cc.WANDB_BASE_URL = WANDB_BASE_URL  # common.py uses it for every run and artifact
if USE_WANDB and not os.environ.get("WANDB_API_KEY"):
    print("No W&B key found: running without W&B (results are still saved locally).")
    USE_WANDB = False
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "MegaDetector on 203k images needs a GPU (Molab GPU runtime)"

SHARD_ID, N_SHARDS = cc.parse_shard(SHARD)
WORK_DIR = cc.work_root()
BOX_DIR = WORK_DIR / "boxes"
CHUNK_DIR = BOX_DIR / "chunks"
FIG_DIR = WORK_DIR / "figures"
for _d in (CHUNK_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"shard {SHARD_ID}/{N_SHARDS} | work dir {WORK_DIR} | W&B {USE_WANDB}")

## 1 · Dataset (verified in notebook 00)

In [ ]:
DS_DIR = cc.ensure_dataset()
meta, CLASS_NAMES = cc.load_metadata(DS_DIR)
PROTOCOL = cc.verify_protocol(meta)
# Fixed image order = metadata.csv order, so chunk k is the same images on every machine
ALL_FILES = meta["filename"].tolist()
if MAX_IMAGES:
    ALL_FILES = ALL_FILES[:MAX_IMAGES]
    print(f"TEST MODE: only the first {MAX_IMAGES} images")
ALL_CHUNKS = cc.chunk_ranges(len(ALL_FILES), CHUNK_SIZE)
MY_CHUNKS = cc.chunk_ranges(len(ALL_FILES), CHUNK_SIZE, SHARD_ID, N_SHARDS)
print(f"{len(ALL_FILES):,} images -> {len(ALL_CHUNKS)} chunks of {CHUNK_SIZE}; this shard runs {len(MY_CHUNKS)}")

## 2 · MegaDetector v5a: download, verify, load

In [ ]:
from megadetector.detection.run_detector import load_detector

MD_PATH, MD_INFO = cc.download_md_weights(WORK_DIR / "models")
DETECTOR = load_detector(str(MD_PATH), force_cpu=not torch.cuda.is_available())
MD_INFO.update(
    {
        "megadetector_package": pkg_metadata.version("megadetector"),
        "torch": torch.__version__,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "batch_size": BATCH_SIZE,
        "chunk_size": CHUNK_SIZE,
        "n_images": len(ALL_FILES),
        "metadata_md5": cc.file_md5(DS_DIR / "metadata.csv"),
    }
)
print(json.dumps(MD_INFO, indent=2))

## 3 · Smoke test on 8 training photos

Checks the output format and draws the boxes (confidence ≥ 0.1) before the long run.

In [ ]:
def draw_boxes(ax, img, boxes, min_conf=0.1):
    # boxes: rows of the box table for one photo; draws normalised [x, y, w, h] boxes
    ax.imshow(img)
    ax.axis("off")
    colours = {"animal": "lime", "person": "red", "vehicle": "orange"}
    for _, b in boxes[(boxes["det_rank"] >= 0) & (boxes["conf"] >= min_conf)].iterrows():
        ax.add_patch(
            Rectangle(
                (b["x"] * img.width, b["y"] * img.height),
                b["w"] * img.width,
                b["h"] * img.height,
                fill=False,
                lw=1.5,
                color=colours.get(b["category"], "cyan"),
            )
        )
        ax.text(b["x"] * img.width, b["y"] * img.height, f"{b['category'][0]} {b['conf']:.2f}",
                color="black", fontsize=6, backgroundcolor=colours.get(b["category"], "cyan"))


_smoke = meta[meta["split"] == "train"].sample(8, random_state=SEED)
_t0 = time.time()
_res = cc.run_md_batches(DETECTOR, [cc.image_path(DS_DIR, _f) for _f in _smoke["filename"]],
                         batch_size=BATCH_SIZE, threads=LOADER_THREADS)
SMOKE_BOXES = pd.DataFrame(
    [_r for _f, _x in zip(_smoke["filename"], _res) for _r in cc.detection_rows(_f, _x)],
    columns=cc.BOX_COLUMNS,
)
assert set(SMOKE_BOXES["filename"]) == set(_smoke["filename"]), "every photo must have a row"
print(f"8 photos in {time.time() - _t0:.1f}s (includes warm-up)")
_fig, _axes = plt.subplots(2, 4, figsize=(14, 5.5))
for _ax, (_, _row) in zip(_axes.ravel(), _smoke.iterrows()):
    with Image.open(cc.image_path(DS_DIR, _row["filename"])) as _img:
        draw_boxes(_ax, _img.convert("RGB"), SMOKE_BOXES[SMOKE_BOXES["filename"] == _row["filename"]])
    _ax.set_title(f"{_row['class_name'][:26]} · cam {_row['camera']}", fontsize=7)
_fig.suptitle("Smoke test: MegaDetector boxes with confidence >= 0.1 (green = animal)")
_fig.tight_layout()
_fig

## 4 · Timing estimate

Runs `TIMING_IMAGES` (256) training photos (not saved) to measure throughput and estimate this shard's run time.

In [ ]:
_sample = meta[meta["split"] == "train"].sample(min(TIMING_IMAGES, len(meta)), random_state=SEED + 1)
_t0 = time.time()
cc.run_md_batches(DETECTOR, [cc.image_path(DS_DIR, _f) for _f in _sample["filename"]],
                  batch_size=BATCH_SIZE, threads=LOADER_THREADS)
IMAGES_PER_SECOND = len(_sample) / (time.time() - _t0)
_todo = sum(_stop - _start for _cid, _start, _stop in MY_CHUNKS
            if not (CHUNK_DIR / f"chunk_{_cid:04d}.csv.gz").is_file())
print(f"{IMAGES_PER_SECOND:.1f} images/s -> this shard has {_todo:,} images left "
      f"≈ {_todo / IMAGES_PER_SECOND / 3600:.1f} h")
if _todo / IMAGES_PER_SECOND > 11 * 3600:
    print("WARNING: longer than one 12 h Molab session. Split the work: SHARD = '0/2' and '1/2'.")

## 5 · The full run (resumable)

Each chunk is written to `boxes/chunks/chunk_NNNN.csv.gz` when it finishes. Rerunning skips
finished chunks.

In [ ]:
RUN = cc.wandb_run(
    f"cropclip-01-megadetector-shard{SHARD_ID}of{N_SHARDS}",
    "megadetector",
    {"notebook": "01_boxes", "shard": SHARD, **MD_INFO},
    enabled=USE_WANDB,
)


def chunk_path(cid):
    return CHUNK_DIR / f"chunk_{cid:04d}.csv.gz"


_t_start, _n_done = time.time(), 0
for _cid, _start, _stop in tqdm(MY_CHUNKS, desc=f"chunks (shard {SHARD_ID}/{N_SHARDS})"):
    if chunk_path(_cid).is_file():
        continue
    _files = ALL_FILES[_start:_stop]
    _res = cc.run_md_batches(DETECTOR, [cc.image_path(DS_DIR, _f) for _f in _files],
                             batch_size=BATCH_SIZE, threads=LOADER_THREADS)
    _df = pd.DataFrame([_r for _f, _x in zip(_files, _res) for _r in cc.detection_rows(_f, _x)],
                       columns=cc.BOX_COLUMNS)
    assert _df["filename"].nunique() == len(_files), "every photo in the chunk must have a row"
    _tmp = chunk_path(_cid).with_suffix(".tmp")
    _df.to_csv(_tmp, index=False, compression="gzip")
    _tmp.rename(chunk_path(_cid))  # appears only when complete
    _n_done += len(_files)
    if RUN is not None:
        RUN.log({"md/images_done": _n_done, "md/images_per_s": _n_done / (time.time() - _t_start),
                 "md/chunk": _cid})
MY_CHUNKS_DONE = [_cid for _cid, _, _ in MY_CHUNKS if chunk_path(_cid).is_file()]
print(f"shard {SHARD_ID}/{N_SHARDS}: {len(MY_CHUNKS_DONE)}/{len(MY_CHUNKS)} chunks done; "
      f"{_n_done:,} images this session")
if N_SHARDS > 1 and len(MY_CHUNKS_DONE) == len(MY_CHUNKS):
    cc.log_artifact(RUN, f"megadetector-v5a-boxes-shard{SHARD_ID}of{N_SHARDS}", "boxes-shard",
                    [chunk_path(_c) for _c in MY_CHUNKS_DONE], MD_INFO)

## 6 · Merge all chunks and upload

With several shards, this downloads the other shards' artifacts first. If some are not finished
yet, the cell stops with a message: rerun it once your teammates are done.

In [ ]:
_missing = [c for c, _, _ in ALL_CHUNKS if not chunk_path(c).is_file()]
if _missing and N_SHARDS > 1 and USE_WANDB:
    for _s in range(N_SHARDS):
        if _s == SHARD_ID:
            continue
        try:
            cc.fetch_artifact(f"megadetector-v5a-boxes-shard{_s}of{N_SHARDS}", CHUNK_DIR, run=RUN)
        except Exception as _err:
            print(f"shard {_s}: not available yet ({type(_err).__name__})")
    _missing = [c for c, _, _ in ALL_CHUNKS if not chunk_path(c).is_file()]
if _missing:
    raise RuntimeError(f"{len(_missing)} chunks missing (e.g. {_missing[:5]}): finish the other shards, then rerun")

BOXES = pd.concat([cc.read_boxes(chunk_path(c)) for c, _, _ in ALL_CHUNKS], ignore_index=True)
assert BOXES["filename"].nunique() == len(ALL_FILES), "every image must appear in the box table"
assert set(BOXES["filename"]) == set(ALL_FILES)
BOXES_PATH = BOX_DIR / "boxes_all.csv.gz"
BOXES.to_csv(BOXES_PATH, index=False, compression="gzip")
(BOX_DIR / "megadetector_run.json").write_text(json.dumps(MD_INFO, indent=2))
_n_fail = int(BOXES.loc[BOXES["failure"] != "", "filename"].nunique())
print(f"{len(BOXES):,} rows for {len(ALL_FILES):,} images | images that failed to load: {_n_fail}")
if not MAX_IMAGES:
    cc.log_artifact(RUN, "megadetector-v5a-boxes", "boxes",
                    [BOXES_PATH, BOX_DIR / "megadetector_run.json"], {**MD_INFO, "failures": _n_fail})

## 7 · Box statistics (train / ID val / OOD val only)

These figures help choose the cutoff τ_det and padding later. Infrared flags come from notebook 00's
image statistics (computed here for the needed photos if they are missing).

In [ ]:
_stats_path = WORK_DIR / "image_stats.csv.gz"
if not _stats_path.is_file() and USE_WANDB:
    try:
        cc.fetch_artifact("iwildcam-image-stats", WORK_DIR, run=RUN)
    except Exception as _err:
        print("no image-stats artifact:", type(_err).__name__)
_dev = meta[meta["split"].isin(["train", "id_val", "val"]) & meta["filename"].isin(ALL_FILES)]
if _stats_path.is_file():
    _stats = pd.read_csv(_stats_path)[["filename", "ir", "width", "height"]]
else:
    _stats = cc.compute_image_stats([cc.image_path(DS_DIR, f) for f in _dev["filename"]], workers=8)
    _stats = _stats.assign(filename=_dev["filename"].to_numpy())[["filename", "ir", "width", "height"]]
DEV = (
    _dev.merge(cc.per_image_box_summary(BOXES), on="filename", how="left", validate="one_to_one")
    .merge(_stats, on="filename", how="left", validate="one_to_one")
)
DEV["kind"] = np.where(DEV["is_empty"], "empty photo", "animal photo")
DEV["light"] = np.where(DEV["ir"].fillna(False).astype(bool), "infrared", "day")
print(DEV.groupby(["split", "kind"]).size().unstack().to_string())

In [ ]:
# Highest animal confidence per photo: animal-labelled vs empty-labelled photos
_fig, _axes = plt.subplots(1, 2, figsize=(14, 4))
for _ax, _splits, _title in ((_axes[0], ["train", "id_val"], "Train + ID val"), (_axes[1], ["val"], "OOD val")):
    for _kind, _c in (("animal photo", "C2"), ("empty photo", "C3")):
        _v = DEV.loc[DEV["split"].isin(_splits) & (DEV["kind"] == _kind), "max_animal_conf"]
        _ax.hist(_v, bins=50, range=(0, 1), histtype="step", lw=1.5, color=_c, label=f"{_kind}s ({len(_v):,})")
    _ax.set_yscale("log")
    _ax.set_xlabel("highest MegaDetector animal confidence in the photo")
    _ax.set_title(_title)
    _ax.legend()
_fig.suptitle("Animal photos should sit right, empty photos left: the overlap is where τ_det matters")
_fig.tight_layout()
_fig.savefig(FIG_DIR / "md_max_conf.png", dpi=120, bbox_inches="tight")
_fig

In [ ]:
# Detector miss rate and false-alarm rate as the cutoff changes, day vs infrared
TAUS = [0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
_rows = []
for _split in ("train", "val"):
    for _light in ("day", "infrared"):
        _part = DEV[(DEV["split"] == _split) & (DEV["light"] == _light)]
        _an, _em = _part[_part["kind"] == "animal photo"], _part[_part["kind"] == "empty photo"]
        for _t in TAUS:
            _rows.append({
                "split": _split, "light": _light, "tau": _t,
                "miss_rate": float((_an["max_animal_conf"] < _t).mean()) if len(_an) else np.nan,
                "false_alarm_rate": float((_em["max_animal_conf"] >= _t).mean()) if len(_em) else np.nan,
            })
TAU_TABLE = pd.DataFrame(_rows)
_fig, _axes = plt.subplots(1, 2, figsize=(14, 4))
for _ax, _split in zip(_axes, ("train", "val")):
    for _light, _ls in (("day", "-"), ("infrared", "--")):
        _p = TAU_TABLE[(TAU_TABLE["split"] == _split) & (TAU_TABLE["light"] == _light)]
        _ax.plot(_p["tau"], 100 * _p["miss_rate"], _ls, color="C2", marker="o", ms=3, label=f"animals missed ({_light})")
        _ax.plot(_p["tau"], 100 * _p["false_alarm_rate"], _ls, color="C3", marker="o", ms=3, label=f"empties with a box ({_light})")
    _ax.set_xscale("log")
    _ax.set_xlabel("τ_det (log scale)")
    _ax.set_ylabel("% of photos")
    _ax.set_title(cc.SPLIT_LABELS[_split])
    _ax.legend(fontsize=7)
_fig.tight_layout()
_fig.savefig(FIG_DIR / "md_tau_tradeoff.png", dpi=120, bbox_inches="tight")
_fig

In [ ]:
# Table: the same trade-off at a few cutoffs, per development split
_rows = []
for _split in ("train", "id_val", "val"):
    _part = DEV[DEV["split"] == _split]
    _an, _em = _part[_part["kind"] == "animal photo"], _part[_part["kind"] == "empty photo"]
    for _t in (0.05, 0.1, 0.2, 0.3, 0.5):
        _rows.append({"split": cc.SPLIT_LABELS[_split], "tau": _t,
                      "% animal photos with a box": round(100 * float((_an["max_animal_conf"] >= _t).mean()), 1),
                      "% empty photos with a box": round(100 * float((_em["max_animal_conf"] >= _t).mean()), 1)})
TAU_SUMMARY = pd.DataFrame(_rows)
TAU_SUMMARY.pivot(index="tau", columns="split")

In [ ]:
# Boxes per animal photo, box size, and box categories (confidence >= 0.2, v5a's typical cutoff)
_dev_boxes = BOXES[BOXES["filename"].isin(DEV["filename"]) & (BOXES["det_rank"] >= 0) & (BOXES["conf"] >= 0.2)]
_dev_boxes = _dev_boxes.merge(DEV[["filename", "kind", "light"]], on="filename")
_fig, _axes = plt.subplots(1, 3, figsize=(17, 4))
_animal_boxes = _dev_boxes[_dev_boxes["category"] == "animal"]
_per_photo = _animal_boxes[_animal_boxes["kind"] == "animal photo"].groupby("filename").size()
_per_photo = _per_photo.reindex(DEV.loc[DEV["kind"] == "animal photo", "filename"], fill_value=0)
_axes[0].hist(_per_photo.clip(upper=10), bins=range(0, 12), color="C0")
_axes[0].set_xlabel("animal boxes per animal photo (clipped at 10)")
_axes[0].set_title("Boxes per photo (conf >= 0.2)")
_area = (_animal_boxes["w"] * _animal_boxes["h"]).clip(lower=1e-5)
for _light in ("day", "infrared"):
    _a = _area[_animal_boxes["light"] == _light]
    if len(_a):
        _axes[1].hist(np.log10(_a), bins=40, histtype="step", density=True, label=_light)
_axes[1].set_xlabel("log10(box area / image area)")
_axes[1].set_title("Animal box size")
_axes[1].legend()
_counts = _dev_boxes.groupby(["category", "kind"]).size().unstack(fill_value=0)
if len(_counts):
    _counts.plot.bar(ax=_axes[2], rot=0)
_axes[2].set_title("Box categories")
_fig.tight_layout()
_fig.savefig(FIG_DIR / "md_box_stats.png", dpi=120, bbox_inches="tight")
_fig

## 8 · Preview grids (train / OOD val)

Boxes with confidence ≥ 0.1. Look for: missed animals (small, dark, partly visible), false boxes
on empty photos (logs, rocks, shadows), and how tight the boxes are.

In [ ]:
def box_grid(rows, title, name, ncols=4, min_conf=0.1):
    rows = rows.reset_index(drop=True)
    nrows = max(1, int(np.ceil(len(rows) / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.9 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        with Image.open(cc.image_path(DS_DIR, r["filename"])) as img:
            draw_boxes(ax, img.convert("RGB"), BOXES[BOXES["filename"] == r["filename"]], min_conf)
        ax.set_title(f"{r['class_name'][:24]} · {cc.SPLIT_LABELS[r['split']]} · {r['light']}\n"
                     f"max animal conf {r['max_animal_conf']:.2f}", fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"md_{name}.png", dpi=110, bbox_inches="tight")
    return fig


def pick(rows, n):
    # up to n random rows (fewer if the group is small)
    return rows.sample(min(n, len(rows)), random_state=SEED)


_an = DEV[DEV["kind"] == "animal photo"]
box_grid(pd.concat([pick(_an[_an["light"] == "day"], 4), pick(_an[_an["light"] == "infrared"], 4)]),
         "Animal photos: day (top) vs infrared (bottom)", "grid_animals")

In [ ]:
_em = DEV[(DEV["kind"] == "empty photo") & (DEV["max_animal_conf"] >= 0.1)]
box_grid(pick(_em.sort_values("max_animal_conf", ascending=False).head(40), 8),
         "Empty photos where MegaDetector still finds an 'animal' (false alarms)", "grid_false_alarms")

In [ ]:
_missed = DEV[(DEV["kind"] == "animal photo") & (DEV["max_animal_conf"] < 0.1)]
print(f"animal photos with no box >= 0.1: {len(_missed):,} of {int((DEV['kind'] == 'animal photo').sum()):,}")
box_grid(pick(_missed, 8), "Animal photos with no box >= 0.1 (misses, or frames without a visible animal)", "grid_misses")

In [ ]:
# What the crop classifier will see: top animal box as a square crop with 15% padding
_top = BOXES[(BOXES["category"] == "animal") & (BOXES["conf"] >= 0.2)].sort_values("conf", ascending=False)
_top = _top.drop_duplicates("filename").merge(DEV[DEV["kind"] == "animal photo"], on="filename")
_fig, _axes = plt.subplots(2, 6, figsize=(15, 5.6), squeeze=False)
for _ax in _axes.ravel():
    _ax.axis("off")
for _ax, (_, _r) in zip(_axes.ravel(), pick(_top, 12).iterrows()):
    with Image.open(cc.image_path(DS_DIR, _r["filename"])) as _img:
        _img = _img.convert("RGB")
        _box = cc.square_crop_box((_r["x"], _r["y"], _r["w"], _r["h"]), _img.width, _img.height, pad=0.15)
        _ax.imshow(_img.crop(_box).resize((224, 224)))
    _ax.set_title(f"{_r['class_name'][:20]}\n{_r['light']} · conf {_r['conf']:.2f}", fontsize=7)
_fig.suptitle("Square crops (15% padding) of the most confident animal box")
_fig.tight_layout()
_fig.savefig(FIG_DIR / "md_grid_crops.png", dpi=110, bbox_inches="tight")
_fig

## 9 · Summary

In [ ]:
_val = DEV[DEV["split"] == "val"]
BOX_SUMMARY = {
    **MD_INFO,
    "n_box_rows": int(len(BOXES)),
    "images_failed": int(BOXES.loc[BOXES["failure"] != "", "filename"].nunique()),
    "ood_val_animal_photos_with_box_at_0.2": round(float((_val.loc[_val["kind"] == "animal photo", "max_animal_conf"] >= 0.2).mean()), 4),
    "ood_val_empty_photos_with_box_at_0.2": round(float((_val.loc[_val["kind"] == "empty photo", "max_animal_conf"] >= 0.2).mean()), 4),
    "git_commit": cc.git_commit(CODE_DIR),
}
(BOX_DIR / "box_summary.json").write_text(json.dumps(BOX_SUMMARY, indent=2))
if RUN is not None:
    import wandb as _wandb

    for _png in sorted(FIG_DIR.glob("md_*.png")):
        RUN.log({f"md/{_png.stem}": _wandb.Image(str(_png))})
    RUN.summary.update({k: v for k, v in BOX_SUMMARY.items() if isinstance(v, (int, float, str))})
    RUN.finish()
BOX_SUMMARY